In [1]:
# uploaded file
from google.colab import files

uploaded = files.upload()


Saving metr-la.csv to metr-la.csv


In [2]:
#  reading csv
import pandas as pd

filename = next(iter(uploaded))

print("Uploaded file:", filename)

df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
display(df.head())


Uploaded file: metr-la.csv
Dataset shape: (34272, 208)


,timestamp,773869,767541,767542,717447,717446,717445,773062,767620,737529,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
0,2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
1,2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2,2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
3,2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
4,2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


In [4]:
# Inspect and fix the index
# If the first column is a timestamp, set it as the index. If it’s an unnamed index column, drop it:
print(df.columns[:5].tolist())

print(df.dtypes.head())
print(df.isna().sum().sum(), "NaNs")
print((df.iloc[:, 1:] == 0).sum().sum(), "zeros")

['timestamp', '773869', '767541', '767542', '717447']
timestamp     object
773869       float64
767541       float64
767542       float64
717447       float64
dtype: object
0 NaNs
575302 zeros


In [5]:
# If first column is time
df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])
df = df.set_index(df.columns[0]).sort_index()

# If instead it's just an unnamed index column:
# df = df.drop(columns=df.columns[0])

print(df.shape)  # expect (34272, 207)

(34272, 207)


In [6]:
# Handle missing values

# In METR-LA, zeros mean missing sensor readings, not zero speed. Treat them as NaN, then fill
import numpy as np

data = df.copy()
data = data.replace(0, np.nan)
print("Missing %:", data.isna().mean().mean() * 100)

data = data.interpolate(method="linear", limit_direction="both")
data = data.ffill().bfill()

Missing %: 8.109350825676486


In [8]:
# Chronological train/val/test split

# Split in time order (70/10/20), never randomly, to avoid leakage:
values = data.values.astype(np.float32)   # (T, N)
T = values.shape[0]

train_end = int(T * 0.7)
val_end = int(T * 0.8)

train, val, test = values[:train_end], values[train_end:val_end], values[val_end:]

In [9]:
# Normalize using training statistics only
mean, std = train.mean(), train.std()

train_n = (train - mean) / std
val_n   = (val - mean) / std
test_n  = (test - mean) / std

In [10]:
# Create sliding windows

# The standard setup is 12 steps in (1 hour), 12 steps out (1 hour):
def make_windows(x, in_len=12, out_len=12):
    X, Y = [], []
    for i in range(len(x) - in_len - out_len + 1):
        X.append(x[i:i+in_len])
        Y.append(x[i+in_len:i+in_len+out_len])
    return np.array(X), np.array(Y)

X_train, y_train = make_windows(train_n)
X_val, y_val     = make_windows(val_n)
X_test, y_test   = make_windows(test_n)

print(X_train.shape, y_train.shape)  # (samples, 12, 207)

(23967, 12, 207) (23967, 12, 207)


In [11]:
  #Optional extras
#Time-of-day feature (helps a lot):
  tod = (data.index.hour * 60 + data.index.minute) / (24 * 60)

In [16]:
# Split the dataset into training, validation, and testing data.
print("Shape:", data.shape)
print("Remaining missing values:", data.isna().sum().sum())

Shape: (34272, 207)
Remaining missing values: 0


In [13]:
# Normalize the data
# We'll calculate the mean and standard deviation using training data only, then apply those same values to the validation and test data.
import numpy as np

values = data.to_numpy(dtype=np.float32)

T = len(values)

train_end = int(T * 0.7)
val_end = int(T * 0.8)

train = values[:train_end]
val = values[train_end:val_end]
test = values[val_end:]

print("Training data:", train.shape)
print("Validation data:", val.shape)
print("Testing data:", test.shape)

Training data: (23990, 207)
Validation data: (3427, 207)
Testing data: (6855, 207)


In [14]:
# Calculate mean and standard deviation from training data
mean = train.mean(axis=0)
std = train.std(axis=0)

# Avoid division by zero
std[std == 0] = 1

# Normalize all three datasets using training statistics
train_n = (train - mean) / std
val_n = (val - mean) / std
test_n = (test - mean) / std

print("Training normalized:", train_n.shape)
print("Validation normalized:", val_n.shape)
print("Testing normalized:", test_n.shape)

print("Training mean (approx.):", train_n.mean())
print("Training standard deviation (approx.):", train_n.std())

Training normalized: (23990, 207)
Validation normalized: (3427, 207)
Testing normalized: (6855, 207)
Training mean (approx.): -1.1360277e-06
Training standard deviation (approx.): 1.000001


In [15]:
#Create sliding windows
def make_windows(x, in_len=12, out_len=12):
    X, Y = [], []

    for i in range(len(x) - in_len - out_len + 1):
        X.append(x[i:i + in_len])
        Y.append(x[i + in_len:i + in_len + out_len])

    return np.array(X), np.array(Y)


X_train, y_train = make_windows(train_n)
X_val, y_val = make_windows(val_n)
X_test, y_test = make_windows(test_n)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (23967, 12, 207)
y_train: (23967, 12, 207)
X_val: (3404, 12, 207)
y_val: (3404, 12, 207)
X_test: (6832, 12, 207)
y_test: (6832, 12, 207)
